# turkish_distill: build the distillation text mix

Thin notebook, **CPU runtime** (no GPU needed). Builds a mixed distillation corpus (`configs/data_mix_v1.yaml`) from the decontaminated Cosmos chunks and the three QA datasets: a quota of texts per kind (short sentences, paragraphs, informal segments, long chunks, questions, Wikipedia passages), each behind a noise filter (page furniture, cookie banners, spam, foreign-script junk), de-duplicated, and with copies of TR-MTEB test texts removed. All logic lives in `src/turkish_distill/build_mix.py`.

The result goes to `data/mix/mix_v1.jsonl` on Drive, with `mix_v1.mix.json` next to it (rejections per reason, lengths, domains per bucket). Then teacher vectors are encoded for it (`colab_encode.ipynb`, `CONFIG = 'configs/mixabl_bge_mix70k.yaml'`) and the ablation run is trained (`colab_pipeline.ipynb`).

Needs on Drive: `data/corpus_clean.jsonl`, `data/benchmark_texts.jsonl` (+ its `.stats.json`), `data/stage1/pairs_train.jsonl`.

## 1. Setup

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

## 2. Parameters

In [ ]:
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)
if importlib.util.find_spec('turkish_distill') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    sys.path.insert(0, f'{REPO_DIR}/src')

MIX_CONFIG = 'configs/data_mix_v1.yaml'
TARGET_TOTAL = None      # None: the config's target (70,000 texts). A number keeps the shares and rescales the quotas
OUT = f'{DRIVE_ROOT}/data/mix/mix_v1.jsonl'
OVERWRITE = False        # True: rebuild even if OUT exists
FORBIDDEN = f'{DRIVE_ROOT}/data/benchmark_texts.jsonl'   # TR-MTEB test texts; copies of them are dropped

for needed in (f'{DRIVE_ROOT}/data/corpus_clean.jsonl', f'{DRIVE_ROOT}/data/stage1/pairs_train.jsonl', FORBIDDEN):
    if not os.path.exists(needed):
        raise FileNotFoundError(f'{needed} not found')
target_arg = f'--target-total {TARGET_TOTAL}' if TARGET_TOTAL else ''
print(MIX_CONFIG, '->', OUT, '| exists:', os.path.exists(OUT))

## 3. Build

One pass over the Cosmos corpus (1,000,000 chunks) plus the benchmark index; it prints progress every 200,000 rows. The first build of the cleaned corpus took about 9 minutes for the decontamination alone, so expect a comparable time (not measured for this step).

In [ ]:
if os.path.exists(OUT) and not OVERWRITE:
    print('already built:', OUT, '(set OVERWRITE = True to rebuild)')
else:
    !python -m turkish_distill.build_mix --config {MIX_CONFIG} --forbidden {FORBIDDEN} --out {OUT} {target_arg}

## 4. Inspect

The table shows, per bucket, how many texts were wanted, got, how many candidates passed the filters, and how many were dropped as duplicates or benchmark copies. Read the samples: the filters are heuristics, and this is where a bucket that looks wrong shows up.

In [ ]:
import json, random
import pandas as pd

stats = json.load(open(OUT.replace('.jsonl', '.mix.json'), encoding='utf-8'))
table = pd.DataFrame({
    name: {'quota': b['quota'], 'selected': b['selected'], 'candidates': b['candidates_seen_after_filters'],
           'dup': b['dropped_duplicates'], 'benchmark copies': b['dropped_benchmark_copies'],
           'chars p10/p50/p90': '/'.join(map(str, b['chars_p10_p50_p90'] or [])), 'shortfall': b.get('shortfall', 0)}
    for name, b in stats['buckets'].items()}).T
display(table)
print('total texts:', stats['total'], '| seconds:', stats['seconds'])
for name, b in stats['buckets'].items():
    print(f"\n{name}: rejected before sampling ->", dict(sorted(b['rejected_before_sampling'].items(), key=lambda x: -x[1])[:6]))

rows = [json.loads(l) for l in open(OUT, encoding='utf-8')]
rng = random.Random(0)
for name in stats['buckets']:
    pool = [r['text'] for r in rows if r['bucket'] == name]
    print(f'\n--- {name}: {len(pool)} texts')
    for t in rng.sample(pool, min(3, len(pool))):
        print('   ', repr(t[:200]))

## 5. Next

1. Encode the teacher vectors for the mix: `colab_encode.ipynb` with `CONFIG = 'configs/mixabl_bge_mix70k.yaml'` and `TEACHERS = ['bge_m3']` (and `configs/mixabl_bge_cosmos70k.yaml` for the random-Cosmos arm).
2. Train each arm in `colab_pipeline.ipynb` (`CONFIG` = that config, skip the teacher-encoding section) and score the `final/` folders on TR-MTEB. The reference arm (the stage-1 QA texts) is `stage1_single_bge_m3`, already done: 62.72.